In [ ]:
# All imports
import numpy as np
import pandas as pd
from Classes import FileManager as fm
from Classes.DataPlotter import DataPlotter, ms_to_minutes
from Classes import Calculations as calc

# Folder with run data (e.g., SD card)
folder = r"Data"

# Get latest CSV file
latest_file = fm.get_latest_file(folder)

# Create plotter object for graphs
# plotter = DataPlotter(r"D:\d_71.csv")
plotter = DataPlotter(latest_file)

# plotter = DataPlotter(CSV_PATH)
plotter.load_data()

# Auto-trim start based on first sustained rise above speed threshold,
# plus optional fixed trim from end.
AUTO_START_SPEED_KPH = 20.0
AUTO_START_MIN_ABOVE_MS = 0.0  # require >= this long above threshold to avoid spikes/noise
TRIM_END_MS = 1000 * 60 * 0.25

# Constant speed scale factor applied to telemetry speed (1.0 = no scaling)
SPEED_SCALE_FACTOR = 1.07

if plotter.data is None:
    raise ValueError("No telemetry data loaded")

if TRIM_END_MS < 0:
    raise ValueError("TRIM_END_MS must be >= 0")

trim_df = plotter.data.copy().sort_values("Tick").reset_index(drop=True)
t0 = float(trim_df["Tick"].min())
t1 = float(trim_df["Tick"].max())

# Detect first sustained run where speed is above threshold,
# then backtrack to the rise-up from low speed that leads into it.
speed_kph = trim_df["Speed_kph"] if "Speed_kph" in trim_df.columns else (trim_df["Speed"] / 1000.0)
is_above = speed_kph >= float(AUTO_START_SPEED_KPH)
seg_id = (is_above != is_above.shift(fill_value=False)).cumsum()

RAMP_START_LOW_KPH = 0  # "near-zero" threshold for ramp-start detection

keep_start = t0
sustained_found = False
for _, seg in trim_df[is_above].groupby(seg_id[is_above]):
    dur_ms = float(seg["Tick"].iloc[-1] - seg["Tick"].iloc[0])
    if dur_ms >= float(AUTO_START_MIN_ABOVE_MS):
        sustained_found = True

        # First index of this valid >20 kph segment
        cross_idx = int(seg.index[0])

        # Backtrack to last sample at/under near-zero before the crossing,
        # then start from the first sample after that point.
        pre = trim_df.loc[:cross_idx - 1] if cross_idx > 0 else trim_df.iloc[0:0]
        near_zero_pre = pre[speed_kph.loc[pre.index] <= RAMP_START_LOW_KPH]

        if not near_zero_pre.empty:
            nz_idx = int(near_zero_pre.index[-1])
            ramp_idx = min(nz_idx + 1, cross_idx)
            keep_start = float(trim_df.loc[ramp_idx, "Tick"])
        else:
            # If no near-zero sample exists before crossing, fall back to crossing itself.
            keep_start = float(trim_df.loc[cross_idx, "Tick"])

        break

# Fallback: if no sustained segment, use the first threshold crossing.
if not sustained_found and bool(is_above.any()):
    cross_idx = int(trim_df[is_above].index[0])
    pre = trim_df.loc[:cross_idx - 1] if cross_idx > 0 else trim_df.iloc[0:0]
    near_zero_pre = pre[speed_kph.loc[pre.index] <= RAMP_START_LOW_KPH]
    if not near_zero_pre.empty:
        nz_idx = int(near_zero_pre.index[-1])
        ramp_idx = min(nz_idx + 1, cross_idx)
        keep_start = float(trim_df.loc[ramp_idx, "Tick"])
    else:
        keep_start = float(trim_df.loc[cross_idx, "Tick"])

keep_end = t1 - float(TRIM_END_MS)
if keep_end <= keep_start:
    raise ValueError(
        f"Invalid trim window: keep_start={keep_start} ms, keep_end={keep_end} ms"
    )

trim_df = trim_df[(trim_df["Tick"] >= keep_start) & (trim_df["Tick"] <= keep_end)].copy()
if trim_df.empty:
    raise ValueError("Trim removed all rows")

# Rebase time so downstream Tick/TimePlot logic remains consistent.
trim_df["Tick"] = trim_df["Tick"] - trim_df["Tick"].iloc[0]
trim_df["TimePlot"] = ms_to_minutes(trim_df["Tick"])
plotter.data = trim_df.reset_index(drop=True)

print(
    f"Auto start trim: speed>={AUTO_START_SPEED_KPH:.1f} kph, "
    f"min_above={AUTO_START_MIN_ABOVE_MS:.0f} ms, "
    f"ramp_low<={RAMP_START_LOW_KPH:.1f} kph, found={sustained_found}"
)
print(f"Trim config: start={keep_start - t0:.0f} ms (auto), end={TRIM_END_MS:.0f} ms")
print(f"Rows after trim: {len(plotter.data)}")

# Track length from coordinates and constant speed scaling.
track_len_df = pd.read_csv("track_coordinates.csv")
track_lat = track_len_df["Latitude"].to_numpy(dtype=float)
track_lon = track_len_df["Longitude"].to_numpy(dtype=float)
track_lat0 = track_lat.min()
track_lon0 = track_lon.min()
track_lat_ref_rad = np.deg2rad(track_lat.mean())

meters_per_deg_lat = 110_540.0
meters_per_deg_lon = 111_320.0 * np.cos(track_lat_ref_rad)
track_x_m = (track_lon - track_lon0) * meters_per_deg_lon
track_y_m = (track_lat - track_lat0) * meters_per_deg_lat
track_xy_local = np.column_stack((track_x_m, track_y_m))
track_next_xy_local = np.roll(track_xy_local, -1, axis=0)
track_segment_lengths_m = np.linalg.norm(track_next_xy_local - track_xy_local, axis=1)
track_total_length_m = float(track_segment_lengths_m.sum())

scaled_df = plotter.data.copy().sort_values("Tick").reset_index(drop=True)
scaled_df["speed_scale_factor"] = float(SPEED_SCALE_FACTOR)
scaled_df["speed_mps_scaled"] = (scaled_df["Speed"] / 3600.0) * float(SPEED_SCALE_FACTOR)
scaled_df["Speed_scaled"] = scaled_df["Speed"] * float(SPEED_SCALE_FACTOR)
scaled_df["speed_kph_scaled"] = scaled_df["Speed_scaled"] / 1000.0

plotter.data = scaled_df

print(f"Using constant speed scale factor: {float(SPEED_SCALE_FACTOR):.6f}")
TRACK_LEN = track_total_length_m / 1000.0




In [ ]:
# Track projection (lat/lon -> local meters from bottom-left)
import numpy as np
import matplotlib.pyplot as plt

track_df = pd.read_csv("track_coordinates.csv")

lat = track_df["Latitude"].to_numpy(dtype=float)
lon = track_df["Longitude"].to_numpy(dtype=float)

lat0 = lat.min()
lon0 = lon.min()
lat_ref_rad = np.deg2rad(lat.mean())

METERS_PER_DEG_LAT = 110_540.0
METERS_PER_DEG_LON = 111_320.0 * np.cos(lat_ref_rad)

track_df["x_m"] = (lon - lon0) * METERS_PER_DEG_LON
track_df["y_m"] = (lat - lat0) * METERS_PER_DEG_LAT

track_xy = track_df[["x_m", "y_m"]].to_numpy()
next_xy = np.roll(track_xy, -1, axis=0)
segment_lengths_m = np.linalg.norm(next_xy - track_xy, axis=1)
cum_len_m = np.concatenate(([0.0], np.cumsum(segment_lengths_m[:-1])))
track_total_length_m = float(segment_lengths_m.sum())
TRACK_LEN = track_total_length_m / 1000.0

print(f"Track nodes: {len(track_df)}")
print(f"Track length: {track_total_length_m:.2f} m ({TRACK_LEN:.4f} km)")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(track_df["x_m"], track_df["y_m"], lw=1.2, color="tab:blue", label="Track")
ax.scatter(track_df["x_m"].iloc[0], track_df["y_m"].iloc[0], color="tab:red", s=40, label="Start")
ax.set_title("Projected Track Geometry")
ax.set_xlabel("x (m) from bottom-left")
ax.set_ylabel("y (m) from bottom-left")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# Augment telemetry with position along projected track
if plotter.data is None:
    raise ValueError("Load telemetry data first")

run_df = plotter.data.copy()
required_cols = ["Tick", "Speed", "Throttle", "Current", "Voltage"]
missing_cols = [c for c in required_cols if c not in run_df.columns]
if missing_cols:
    raise ValueError(f"Missing telemetry columns: {missing_cols}")

run_df["Tick"] = pd.to_numeric(run_df["Tick"], errors="raise")
run_df["Speed"] = pd.to_numeric(run_df["Speed"], errors="raise")
run_df["Current"] = pd.to_numeric(run_df["Current"], errors="raise")
run_df["Voltage"] = pd.to_numeric(run_df["Voltage"], errors="raise")

speed_source_col = "Speed_scaled"
if speed_source_col not in run_df.columns:
    raise ValueError("Missing Speed_scaled column. Re-run the preprocessing cell that applies SPEED_SCALE_FACTOR.")
run_df["speed_source"] = speed_source_col
run_df["speed_for_distance"] = pd.to_numeric(run_df[speed_source_col], errors="raise")

run_df["dt_s"] = run_df["Tick"].diff().fillna(0.0) / 1000.0
run_df["speed_mps"] = run_df["speed_for_distance"] / 3600.0  # Speed is meters/hour in CSV
run_df["speed_kph"] = run_df["speed_for_distance"] / 1000.0
run_df["ds_m"] = run_df["speed_mps"] * run_df["dt_s"]
run_df["s_m"] = run_df["ds_m"].cumsum()

lap_len_m = float(track_total_length_m)
lap_idx = np.floor(run_df["s_m"].to_numpy(dtype=float) / lap_len_m).astype(int)
run_df["lap_idx"] = lap_idx
run_df["lap_number"] = lap_idx + 1  # Human-friendly lap count starts at 1

# Drop lap 5+ so only relevant laps are analyzed.
run_df = run_df[run_df["lap_number"] < 5].copy()
lap_idx = run_df["lap_idx"].to_numpy(dtype=int)

# Raw modulo distance from track point 0 (the first map coordinate)
s_mod_raw = np.mod(run_df["s_m"].to_numpy(dtype=float), lap_len_m)

# Lock half-lap anchor to where we stop (speed == 0) on the first lap,
# away from start/finish.
STOP_SPEED_MPS = 0.0

first_lap_mask = lap_idx == 0
first_lap_s = s_mod_raw[first_lap_mask]
first_lap_v = run_df.loc[first_lap_mask, "speed_mps"].to_numpy(dtype=float)
mid_window = (first_lap_s >= 0.20 * lap_len_m) & (first_lap_s <= 0.80 * lap_len_m)
first_lap_zero = mid_window & (first_lap_v == STOP_SPEED_MPS)

if np.any(first_lap_zero):
    half_anchor_m = float(np.median(first_lap_s[first_lap_zero]))
elif np.any(mid_window):
    # Fallback only if no exact-zero sample exists in the expected stop region.
    first_lap_stop_idx = np.argmin(first_lap_v[mid_window])
    half_anchor_m = float(first_lap_s[mid_window][first_lap_stop_idx])
else:
    half_anchor_m = lap_len_m / 2.0

# Phase-lock each lap so all laps share the same start and half-stop anchors.
s_mod_locked = s_mod_raw.copy()
eps = 1e-6
for li in np.unique(lap_idx):
    mask = lap_idx == li
    if not np.any(mask):
        continue

    s_lap = s_mod_raw[mask]
    v_lap = run_df.loc[mask, "speed_mps"].to_numpy(dtype=float)
    lap_mid_window = (s_lap >= 0.20 * lap_len_m) & (s_lap <= 0.80 * lap_len_m)

    if not np.any(lap_mid_window):
        continue

    lap_zero = lap_mid_window & (v_lap == STOP_SPEED_MPS)
    if np.any(lap_zero):
        lap_stop_m = float(np.median(s_lap[lap_zero]))
    else:
        # Fallback only if exact-zero sample is missing in this lap.
        lap_stop_idx = np.argmin(v_lap[lap_mid_window])
        lap_stop_m = float(s_lap[lap_mid_window][lap_stop_idx])

    if not (eps < lap_stop_m < lap_len_m - eps and eps < half_anchor_m < lap_len_m - eps):
        continue

    locked = s_lap.copy()
    left = s_lap <= lap_stop_m
    right = ~left

    locked[left] = s_lap[left] * (half_anchor_m / lap_stop_m)
    locked[right] = half_anchor_m + (s_lap[right] - lap_stop_m) * ((lap_len_m - half_anchor_m) / (lap_len_m - lap_stop_m))
    s_mod_locked[mask] = np.clip(locked, 0.0, lap_len_m)

run_df["half_anchor_m"] = half_anchor_m
run_df["s_mod_m"] = s_mod_locked

seg_starts = cum_len_m
seg_ends = cum_len_m + segment_lengths_m
seg_idx = np.searchsorted(seg_ends, s_mod_locked, side="right")
seg_idx = np.minimum(seg_idx, len(segment_lengths_m) - 1)

local_s = s_mod_locked - seg_starts[seg_idx]
safe_seg_len = np.where(segment_lengths_m[seg_idx] > 0.0, segment_lengths_m[seg_idx], 1.0)
frac = np.clip(local_s / safe_seg_len, 0.0, 1.0)

xy_start = track_xy[seg_idx]
xy_end = next_xy[seg_idx]
car_xy = xy_start + (xy_end - xy_start) * frac[:, None]

run_df["car_x_m"] = car_xy[:, 0]
run_df["car_y_m"] = car_xy[:, 1]
run_df["power_w"] = (run_df["Current"] / 1e3) * (run_df["Voltage"] / 1e3)

plotter.data = run_df

print("Added columns:", ["car_x_m", "car_y_m", "speed_mps", "speed_kph", "power_w", "lap_idx", "lap_number", "half_anchor_m"])
print(plotter.data[["Tick", "lap_number", "speed_kph", "car_x_m", "car_y_m", "power_w"]].head())
print(f"Half anchor (from lap 1 stop): {half_anchor_m:.2f} m")
print(f"Detected laps: {int(plotter.data['lap_number'].max())}")
print(f"Distance speed source: {speed_source_col}")



In [ ]:
# Per-lap metric overlays on the projected track map
if plotter.data is None:
    raise ValueError("Load and augment telemetry data first")
for col in ["car_x_m", "car_y_m", "speed_kph", "Throttle", "power_w", "lap_number"]:
    if col not in plotter.data.columns:
        raise ValueError(f"Expected augmented column missing: {col}")

vis_df = plotter.data
metrics = [
    ("speed_kph", "Speed (km/h)", "viridis"),
    ("power_w", "Power (kW)", "inferno"),
]

lap_numbers = sorted(vis_df["lap_number"].astype(int).unique().tolist())
n_laps = len(lap_numbers)
n_rows = len(metrics)

fig, axs = plt.subplots(
    n_rows,
    n_laps,
    figsize=(4.2 * n_laps, 3.8 * n_rows),
    constrained_layout=True,
    squeeze=False,
)

x_lim = (track_df["x_m"].min(), track_df["x_m"].max())
y_lim = (track_df["y_m"].min(), track_df["y_m"].max())

for col_idx, lap in enumerate(lap_numbers):
    lap_df = vis_df[vis_df["lap_number"] == lap]
    x = lap_df["car_x_m"].to_numpy()
    y = lap_df["car_y_m"].to_numpy()

    for row_idx, (metric, title, cmap) in enumerate(metrics):
        ax = axs[row_idx, col_idx]
        ax.plot(track_df["x_m"], track_df["y_m"], color="0.75", lw=1.0)
        sc = ax.scatter(x, y, c=lap_df[metric].to_numpy(), s=6, cmap=cmap, alpha=0.9)
        cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
        cb.set_label(title)

        if row_idx == 0:
            ax.set_title(f"Lap {lap}")
        if col_idx == 0:
            ax.set_ylabel(f"{title}\ny (m)")
        else:
            ax.set_ylabel("y (m)")

        ax.set_xlabel("x (m)")
        ax.set_xlim(*x_lim)
        ax.set_ylim(*y_lim)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
# Visual debug: stop marker alignment by lap (robust pause detection)
if plotter.data is None:
    raise ValueError("Load and augment telemetry data first")

for col in ["Tick", "lap_number", "speed_mps", "car_x_m", "car_y_m", "s_mod_m"]:
    if col not in plotter.data.columns:
        raise ValueError(f"Expected augmented column missing: {col}")

vis = plotter.data.copy().sort_values("Tick").reset_index(drop=True)
lap_numbers = sorted(vis["lap_number"].astype(int).unique().tolist())

# Use the same middle window as the anchoring logic to avoid start/finish low-speed artifacts
lap_len_m = float(track_total_length_m)
mid_lo = 0.20 * lap_len_m
mid_hi = 0.80 * lap_len_m

# Pause detection thresholds
# Exact stop means speed is exactly 0 m/s (clean stop telemetry).
stop_speed_mps = 0.0
min_pause_s = 2  # require sustained pause duration

stop_rows = []
for lap in lap_numbers:
    lap_df = vis[vis["lap_number"] == lap].copy()
    lap_df = lap_df[(lap_df["s_mod_m"] >= mid_lo) & (lap_df["s_mod_m"] <= mid_hi)].copy()
    if lap_df.empty:
        continue

    lap_df = lap_df.sort_values("Tick")
    is_zeroish = lap_df["speed_mps"] == stop_speed_mps

    # Build contiguous low-speed segments
    seg_id = (is_zeroish != is_zeroish.shift(fill_value=False)).cumsum()
    pause_candidates = []
    for _, seg in lap_df[is_zeroish].groupby(seg_id[is_zeroish]):
        if seg.empty:
            continue
        dur_s = (seg["Tick"].iloc[-1] - seg["Tick"].iloc[0]) / 1000.0
        pause_candidates.append((dur_s, seg))

    use_fallback = True
    if pause_candidates:
        # Keep only sustained pauses, then choose the longest one
        sustained = [x for x in pause_candidates if x[0] >= min_pause_s]
        if sustained:
            use_fallback = False
            dur_s, best_seg = max(sustained, key=lambda x: x[0])

            # Representative stop point: midpoint in time of the sustained pause
            t_mid = 0.5 * (best_seg["Tick"].iloc[0] + best_seg["Tick"].iloc[-1])
            rep_idx = (best_seg["Tick"] - t_mid).abs().idxmin()
            stop_rows.append({
                "lap_number": int(lap),
                "car_x_m": float(vis.loc[rep_idx, "car_x_m"]),
                "car_y_m": float(vis.loc[rep_idx, "car_y_m"]),
                "speed_mps": float(vis.loc[rep_idx, "speed_mps"]),
                "s_mod_m": float(vis.loc[rep_idx, "s_mod_m"]),
                "pause_duration_s": float(dur_s),
                "method": "long_pause",
            })

    if use_fallback:
        # Fallback only when no sustained pause exists in this lap window
        idx = lap_df["speed_mps"].idxmin()
        stop_rows.append({
            "lap_number": int(lap),
            "car_x_m": float(vis.loc[idx, "car_x_m"]),
            "car_y_m": float(vis.loc[idx, "car_y_m"]),
            "speed_mps": float(vis.loc[idx, "speed_mps"]),
            "s_mod_m": float(vis.loc[idx, "s_mod_m"]),
            "pause_duration_s": 0.0,
            "method": "fallback_min_speed",
        })

stop_df = pd.DataFrame(stop_rows).sort_values("lap_number") if stop_rows else pd.DataFrame()

if stop_df.empty:
    raise ValueError("No stop candidates found. Check thresholds/window and data quality.")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(track_df["x_m"], track_df["y_m"], color="0.75", lw=1.2, label="Track")

# Lap-1 reference stop
ref = stop_df.iloc[0]
ax.scatter(ref["car_x_m"], ref["car_y_m"], s=140, marker="*", color="black", label="Lap 1 reference stop")

# Plot one stop marker per lap
sc = ax.scatter(
    stop_df["car_x_m"],
    stop_df["car_y_m"],
    c=stop_df["lap_number"],
    cmap="tab10",
    s=70,
    edgecolor="black",
    linewidth=0.6,
    label="Detected stop per lap",
)

for _, r in stop_df.iterrows():
    tag = f"L{int(r['lap_number'])}"
    if r["method"] != "long_pause":
        tag += "*"
    ax.text(r["car_x_m"], r["car_y_m"], tag, fontsize=8, ha="left", va="bottom")

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Lap number")

ax.set_aspect("equal", adjustable="box")
ax.set_title("Stop Alignment Check Across Laps")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.grid(alpha=0.25)
ax.legend(loc="best")
plt.show()

print("Note: labels with * used fallback (no sustained pause met threshold).")
print(stop_df[["lap_number", "s_mod_m", "speed_mps", "pause_duration_s", "method"]].to_string(index=False))

In [ ]:
# Plot all columns
plotter.plot_all(smooth_window=0)

In [ ]:
# Efficiency from augmented telemetry (exact path + integrated energy)
if plotter.data is None:
    raise ValueError("Load and augment telemetry data first")

required = ["Tick", "power_w", "car_x_m", "car_y_m"]
missing = [c for c in required if c not in plotter.data.columns]
if missing:
    raise ValueError(f"Missing required augmented columns: {missing}")

run_df = plotter.data.sort_values("Tick").reset_index(drop=True).copy()

# Exact traveled distance from projected map coordinates
run_df["dx_m"] = run_df["car_x_m"].diff().fillna(0.0)
run_df["dy_m"] = run_df["car_y_m"].diff().fillna(0.0)
run_df["ds_exact_m"] = np.hypot(run_df["dx_m"], run_df["dy_m"])
run_df["s_exact_m"] = run_df["ds_exact_m"].cumsum()

# Energy integration from power profile
run_df["t_s"] = run_df["Tick"] / 1000.0
total_energy_j = float(np.trapezoid(run_df["power_w"], x=run_df["t_s"]))
total_energy_kwh = total_energy_j / 3_600_000.0

total_distance_km = run_df["ds_exact_m"].sum() / 1000.0
total_eff_km_per_kwh = total_distance_km / total_energy_kwh if total_energy_kwh > 0 else np.nan

print(f"Total exact traveled distance: {total_distance_km:.4f} km")
print(f"Total integrated energy: {total_energy_j:.1f} J ({total_energy_kwh:.6f} kWh)")
print(f"Total efficiency: {total_eff_km_per_kwh:.2f} km/kWh")

# Segment efficiencies by lap and half-lap
lap_len_m = float(track_total_length_m)
half_anchor_m = float(run_df["half_anchor_m"].iloc[0]) if "half_anchor_m" in run_df.columns else (lap_len_m / 2.0)

run_df["lap_seg_idx"] = np.floor(run_df["s_exact_m"] / lap_len_m).astype(int)
run_df["s_exact_mod_m"] = np.mod(run_df["s_exact_m"], lap_len_m)
run_df["half_seg_idx"] = run_df["lap_seg_idx"] * 2 + (run_df["s_exact_mod_m"] >= half_anchor_m).astype(int)

lap_rows = []
for seg_idx, seg in run_df.groupby("lap_seg_idx", sort=True):
    if len(seg) < 2:
        continue
    dist_km = seg["ds_exact_m"].sum() / 1000.0
    e_j = float(np.trapezoid(seg["power_w"], x=seg["t_s"]))
    e_kwh = e_j / 3_600_000.0
    eff = dist_km / e_kwh if e_kwh > 0 else np.nan
    lap_rows.append({
        "segment": f"Lap {seg_idx + 1}",
        "distance_km": dist_km,
        "energy_j": e_j,
        "eff_km_per_kwh": eff,
    })

half_rows = []
for seg_idx, seg in run_df.groupby("half_seg_idx", sort=True):
    if len(seg) < 2:
        continue
    lap_num = (seg_idx // 2) + 1
    half_num = (seg_idx % 2) + 1
    dist_km = seg["ds_exact_m"].sum() / 1000.0
    e_j = float(np.trapezoid(seg["power_w"], x=seg["t_s"]))
    e_kwh = e_j / 3_600_000.0
    eff = dist_km / e_kwh if e_kwh > 0 else np.nan
    half_rows.append({
        "segment": f"L{lap_num}-H{half_num}",
        "distance_km": dist_km,
        "energy_j": e_j,
        "eff_km_per_kwh": eff,
    })

lap_eff_df = pd.DataFrame(lap_rows)
half_eff_df = pd.DataFrame(half_rows)

print("\nLap efficiencies:")
print(lap_eff_df[["segment", "distance_km", "energy_j", "eff_km_per_kwh"]])

print("\nHalf-lap efficiencies:")
print(half_eff_df[["segment", "distance_km", "energy_j", "eff_km_per_kwh"]])

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)

lap_bars = axes[0].bar(lap_eff_df["segment"], lap_eff_df["eff_km_per_kwh"], color="tab:blue", alpha=0.85)
axes[0].set_title("Efficiency by Lap")
axes[0].set_ylabel("km/kWh")
axes[0].grid(axis="y", alpha=0.3)
for bar, val in zip(lap_bars, lap_eff_df["eff_km_per_kwh"]):
    if pd.notna(val):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f} km/kWh",
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=0,
        )

half_bars = axes[1].bar(half_eff_df["segment"], half_eff_df["eff_km_per_kwh"], color="tab:orange", alpha=0.85)
axes[1].set_title("Efficiency by Half-Lap")
axes[1].set_ylabel("km/kWh")
axes[1].set_xlabel("Segment")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", alpha=0.3)
for bar, val in zip(half_bars, half_eff_df["eff_km_per_kwh"]):
    if pd.notna(val):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f} km/kWh",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=0,
        )

plt.show()

# Persist augmented distance columns back into plotter state
plotter.data = run_df

In [ ]:
# Optional helper cross-check (same integrated energy source)
calc.get_total_energy(plotter.data)

In [ ]:
# Animated telemetry playback over OpenStreetMap background
# Run this cell after the augmentation cell if you want true telemetry playback.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from io import BytesIO
import math
import matplotlib

matplotlib.rcParams['animation.embed_limit'] = 2**128

# Track coordinates (used for map extent + projection inversion)
track_df = pd.read_csv("track_coordinates.csv")
track_lat = track_df["Latitude"].to_numpy(dtype=float)
track_lon = track_df["Longitude"].to_numpy(dtype=float)

lat0 = track_lat.min()
lon0 = track_lon.min()
lat_ref_rad = np.deg2rad(track_lat.mean())
METERS_PER_DEG_LAT = 111_132.0
METERS_PER_DEG_LON = 111_320.0 * np.cos(lat_ref_rad)

# Prefer augmented telemetry when available
if "plotter" in globals() and getattr(plotter, "data", None) is not None:
    candidate = plotter.data.copy().sort_values("Tick").reset_index(drop=True)
else:
    candidate = pd.DataFrame()

if not candidate.empty and {"car_x_m", "car_y_m"}.issubset(candidate.columns):
    # Convert augmented local-meter coordinates back to geographic coords for map overlay
    anim_lat_full = lat0 + candidate["car_y_m"].to_numpy(dtype=float) / METERS_PER_DEG_LAT
    anim_lon_full = lon0 + candidate["car_x_m"].to_numpy(dtype=float) / METERS_PER_DEG_LON

    if "Tick" in candidate.columns and candidate["Tick"].notna().any():
        tick_s = candidate["Tick"].to_numpy(dtype=float)
    else:
        tick_s = np.arange(len(anim_lat_full), dtype=float)

    source_label = "Augmented telemetry"
else:
    # Fallback: animate through the reference track points
    anim_lat_full = track_lat.copy()
    anim_lon_full = track_lon.copy()
    tick_s = np.arange(len(anim_lat_full), dtype=float)
    source_label = "Track reference points (fallback)"

# Build padded map bounding box
lat_pad = (track_lat.max() - track_lat.min()) * 0.08 or 0.0005
lon_pad = (track_lon.max() - track_lon.min()) * 0.08 or 0.0005
bbox = {
    "left": float(track_lon.min() - lon_pad),
    "right": float(track_lon.max() + lon_pad),
    "bottom": float(track_lat.min() - lat_pad),
    "top": float(track_lat.max() + lat_pad),
}

# Fetch OSM tile mosaic directly (no static-map API dependency)
def deg2tile(lat_deg, lon_deg, zoom):
    lat_rad = math.radians(lat_deg)
    n = 2 ** zoom
    xtile = int((lon_deg + 180.0) / 360.0 * n)
    ytile = int((1.0 - math.log(math.tan(lat_rad) + (1.0 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
    return xtile, ytile


def tile2deg(xtile, ytile, zoom):
    n = 2 ** zoom
    lon_deg = xtile / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1.0 - 2.0 * ytile / n)))
    lat_deg = math.degrees(lat_rad)
    return lat_deg, lon_deg


def choose_zoom(left, bottom, right, top, max_tiles=36):
    # pick highest zoom that keeps requested area to a manageable tile count
    for z in range(18, 5, -1):
        x0, y0 = deg2tile(top, left, z)
        x1, y1 = deg2tile(bottom, right, z)
        tile_count = (abs(x1 - x0) + 1) * (abs(y1 - y0) + 1)
        if tile_count <= max_tiles:
            return z
    return 6


def fetch_tile_mosaic(left, bottom, right, top):
    zoom = choose_zoom(left, bottom, right, top)
    x0, y0 = deg2tile(top, left, zoom)
    x1, y1 = deg2tile(bottom, right, zoom)
    xmin, xmax = sorted((x0, x1))
    ymin, ymax = sorted((y0, y1))

    tile_size = 256
    channels = 4
    mosaic = np.zeros(((ymax - ymin + 1) * tile_size, (xmax - xmin + 1) * tile_size, channels), dtype=np.float32)

    headers = {
        "User-Agent": "UOSM-CleanTelemProcess/0.1 (local notebook visualization)",
    }

    last_exc = None
    fetched = 0
    for xtile in range(xmin, xmax + 1):
        for ytile in range(ymin, ymax + 1):
            tile_url = f"https://tile.openstreetmap.org/{zoom}/{xtile}/{ytile}.png"
            try:
                req = Request(tile_url, headers=headers)
                with urlopen(req, timeout=12) as response:
                    tile_img = plt.imread(BytesIO(response.read()))
                if tile_img.ndim == 2:
                    tile_img = np.stack([tile_img, tile_img, tile_img, np.ones_like(tile_img)], axis=-1)
                elif tile_img.shape[-1] == 3:
                    alpha = np.ones((tile_img.shape[0], tile_img.shape[1], 1), dtype=tile_img.dtype)
                    tile_img = np.concatenate([tile_img, alpha], axis=-1)

                xoff = (xtile - xmin) * tile_size
                yoff = (ytile - ymin) * tile_size
                mosaic[yoff:yoff + tile_size, xoff:xoff + tile_size, :] = tile_img[:, :, :4]
                fetched += 1
            except Exception as exc:
                last_exc = exc

    if fetched == 0:
        raise RuntimeError(last_exc)

    # extent from outer tile edges
    lat_top, lon_left = tile2deg(xmin, ymin, zoom)
    lat_bottom, lon_right = tile2deg(xmax + 1, ymax + 1, zoom)
    extent = [lon_left, lon_right, lat_bottom, lat_top]
    return mosaic, extent


map_img = None
map_extent = [bbox["left"], bbox["right"], bbox["bottom"], bbox["top"]]
try:
    map_img, map_extent = fetch_tile_mosaic(bbox["left"], bbox["bottom"], bbox["right"], bbox["top"])
except Exception as exc:
    print(f"Could not fetch OSM map tiles ({exc}). Rendering without map background.")


def downsample_image_for_animation(img, max_dim=1400):
    """Reduce static background resolution once to speed up frame rendering."""
    if img is None:
        return None
    h, w = img.shape[:2]
    largest_dim = max(h, w)
    if largest_dim <= max_dim:
        return img
    stride = int(np.ceil(largest_dim / max_dim))
    return img[::stride, ::stride]


map_img = downsample_image_for_animation(map_img, max_dim=1400)

# Downsample for smooth rendering while preserving timing spread
max_frames = 900
step = max(1, len(anim_lat_full) // max_frames)
anim_lat = anim_lat_full[::step]
anim_lon = anim_lon_full[::step]
anim_t = tick_s[::step]

# Frame interval in ms (roughly respects telemetry time if Tick exists)
if len(anim_t) > 1:
    dt_ms = np.diff(anim_t) * 1000.0
    interval_ms = int(np.clip(np.nanmedian(dt_ms), 20, 200))
else:
    interval_ms = 45

fig, ax = plt.subplots(figsize=(9, 7))
if map_img is not None:
    ax.imshow(
        map_img,
        extent=map_extent,
        origin="upper",
        interpolation="bilinear",
    )

ax.plot(track_lon, track_lat, color="deepskyblue", linewidth=2.0, alpha=0.7, label="Track")
car_dot, = ax.plot([], [], "o", color="crimson", markersize=8, label="Car", animated=True)
car_tail, = ax.plot([], [], color="crimson", linewidth=2, alpha=0.6, animated=True)
trail_points = 300  # Cap tail history to keep per-frame updates cheap

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Track Playback ({source_label})")
ax.legend(loc="upper right")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(bbox["left"], bbox["right"])
ax.set_ylim(bbox["bottom"], bbox["top"])

ax.text(
    0.01,
    0.01,
    "© OpenStreetMap contributors",
    transform=ax.transAxes,
    fontsize=8,
    color="black",
    bbox={"facecolor": "white", "alpha": 0.6, "edgecolor": "none"},
)


def init():
    car_dot.set_data([], [])
    car_tail.set_data([], [])
    return car_dot, car_tail


def update(frame_idx):
    car_dot.set_data([anim_lon[frame_idx]], [anim_lat[frame_idx]])
    tail_start = max(0, frame_idx - trail_points + 1)
    car_tail.set_data(anim_lon[tail_start: frame_idx + 1], anim_lat[tail_start: frame_idx + 1])
    return car_dot, car_tail

ani = FuncAnimation(
    fig,
    update,
    frames=len(anim_lon),
    init_func=init,
    interval=interval_ms,
    blit=True,
    cache_frame_data=False,
    repeat=True,
)

plt.close(fig)
HTML(ani.to_jshtml())